# AB-DB tutorial: finding, downloading, visualizing and retrieving analysis
***

This Jupyter Notebook will guide you through the process of **finding**, **downloading**, **visualizing** and finally **retrieving analysis results** from the collection of **AB-DB** ligands **MD simulations** and **QM data** available in the **[AB-DB](https://mmb.irbbarcelona.org/ABDB/) database**.

This workflow is based on the **AB-DB database REST API**: https://mmb.irbbarcelona.org/ABDB/rest

Main **AB-DB website**: https://mmb.irbbarcelona.org/ABDB/

***Authors:*** *Giuliano Malloci, Silvia Gervasoni, Adam Hospital, Genís Bayarri*
<br>
***Version:*** *July 2026*

***
## Tutorial steps
 1. [Libraries and Global Variables](#libraries)
 2. [Listing AB-DB Compounds](#listing)
 3. [Browsing AB-DB Database](#browsing)
 4. [Searching & Finding in the AB-DB Database](#searching)
 5. [Physico-chemical descriptors from AB-DB](#descriptors)
 6. [MD data from AB-DB](#md)
    - *Structure downloading*
    - *Structure visualization*
    - *Trajectory downloading*
    - *Trajectory visualization*
    - *Trajectory analyses (MDDB<sup>1</sup>)*
 7. [QM data from AB-DB](#qm)
    - *Download the QM-optimized structure and compare it against the original one*
    - *AMBER force-field parameters generated from the QM calculations*
    - *Geometry optimization and Single-point energy calculation data (ioChem-BD<sup>2</sup>)*
 8. [Questions & Comments](#questions)


<sup>1</sup> https://mdposit.mddbr.eu/
<br>
<sup>2</sup> https://www.iochem-bd.org/
***

<div style="text-align:center;">
<img src="https://upload.wikimedia.org/wikipedia/commons/d/de/Logo_Universit%C3%A0_di_Cagliari.jpg" alt="University of Cagliari logo"
	title="University of Cagliari logo" width="150" style="margin-right: 20px; align: center"/>
<img src="https://mmb.irbbarcelona.org/ABDB/img/home/irb.png" alt="IRB logo"
	title="IRB logo" width="250" style="margin-left: 20px;"/>
</div>

***

In [ ]:
# Only executed when using google colab
import sys
import os
if 'google.colab' in sys.modules:

  try:
      from google.colab import output as _colab_output
      _colab_output.enable_custom_widget_manager()  # must precede nglview import
      get_ipython().run_line_magic('pip', 'install -q "nglview==3.0.8" mdtraj py3Dmol rich')
  except ImportError:
      pass  # not running in Colab

<a id="libraries"></a>
## Libraries and Global Variables  

In [ ]:
# Auxiliary libraries

import urllib.request
import json
import os
from math import floor, ceil
from os.path import exists
from urllib.parse import quote

import ipywidgets
import nglview
#import simpletraj

import pandas as pd

#import plotly
#import plotly.graph_objs as go
#plotly.offline.init_notebook_mode(connected=True)

from rich import print
from rich.panel import Panel

In [ ]:
# URLs to the REST APIs and Web Servers

web_url = 'https://mmb.irbbarcelona.org/ABDB'
api_url = f'{web_url}/api'

mddb_web = 'https://mdposit.mddbr.eu/'

iochem_bd_web = 'https://iochem-bd.org/'

<a id="listing"></a>
## Listing AB-DB Compounds
Listing the **AB-DB collection compounds**.

Exposing the **AB-DB persistent id (FAIR_ID)**, **compound**, **charge**, **family**, **formula**, **inchikey**, **pubchem** and **MDDB accession**.

In [ ]:
compounds_request = f'{api_url}/compounds?limit=1000'

# Query the API
with urllib.request.urlopen(compounds_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

print(json.dumps(parsed_response['data'], indent=2))


<a id="browsing"></a>
## Browsing AB-DB Database

**Browse** the **AB-DB collection** from the **MDDB database**.

In [ ]:
from collections import defaultdict

families = defaultdict(list)

compounds_request = f'{api_url}/compounds?limit=1000'

# Query the API
with urllib.request.urlopen(compounds_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

for entry in parsed_response['data']:
    #compound = entry['Compound']
    family = entry['Family'].title()
    families[family].append(entry)

for family, entries in families.items():
    print(f"=== {family} ===")
    for entry in entries:
        print(f"- Compound: {entry['Compound']}, Charge: {entry['Charge']}, ABDB Accession: {entry['FAIR_ID']}")
    print()


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Dropdown
dropdown = widgets.Dropdown(
    options=sorted(families.keys()),
    description="Family:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
output = widgets.Output()

def show_class(change):
    output.clear_output()
    clase = change['new']
    with output:
        print(f"=== {clase} ===\n")
        for entry in families[clase]:
            print(f"--- {entry['FAIR_ID']} ---")
            print(f"{entry['Compound'].title()} {entry['Charge']}, {entry['Formula']}")
            print(f"{web_url}/compounds/{entry['FAIR_ID']}\n")

dropdown.observe(show_class, names='value')

display(dropdown, output)


<a id="searching"></a>
## Searching & Finding in the AB-DB Database

**Searching** for a particular entry from the **AB-DB collection**.

In [ ]:
# Keyword to search for:
keyword = "aminocoumarins"

# Get a list with all compounds from the specified family from the AB-DB collection
simulations_request = api_url + f'/compounds?family={keyword}'

# Query the API
with urllib.request.urlopen(simulations_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

#print(json.dumps(parsed_response['data'], indent=2))

for entry in parsed_response['data']:
    compound = entry['Compound']
    family = entry['Family']
    charge = entry['Charge']
    print(f"{family.title()}-{compound.title()} (Charge: {charge})")
    print(f"{web_url}/compounds/{entry['FAIR_ID']}\n")

<a id="descriptors"></a>
## Physico-chemical Descriptors from AB-DB

In [ ]:
# Accession to search for:
accession = "ABDB001006"

# Set the descriptors query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
descriptors_query = specific_project_url + '/files/csv'
print('We query the API at ' + descriptors_query)

# Download the file with an arbitrary name
descriptors_filename = f'{accession}_descriptors.csv'

# Query the API
urllib.request.urlretrieve(descriptors_query, descriptors_filename)

if exists(descriptors_filename):
    print(f'File has been downloaded successfully ({descriptors_filename})')

df = pd.read_csv(descriptors_filename)
df.style.set_table_attributes('class="dataframe table table-striped table-bordered"')

<a id="md"></a>
## MD Data from AB-DB

**Downloading** and **visualizing** **MD data** for a particular **AB-DB compound**.

1) **Structure downloading**
2) **Structure visualization**
3) **Trajectory downloading**
4) **Trajectory visualization**
5) **Trajectory analyses (MDDB)**

***
1) **Structure downloading**
***

In [ ]:
# Set the structure query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
structure_query = specific_project_url + '/files/structure.pdb'
print('We query the API at ' + structure_query)

# Download the file with an arbitrary name
structure_filename = f'{accession}_structure.pdb'

# Query the API
urllib.request.urlretrieve(structure_query, structure_filename)

if exists(structure_filename):
    print(f'File has been downloaded successfully ({structure_filename})')

***
2) **Structure visualization**
***

In [ ]:
view = nglview.show_structure_file(structure_filename)
view._remote_call('setSize', target='Widget', args=['','600px'])
view

***
3) **Trajectory downloading**
***

In [ ]:
# Set the trajectory query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
trajectory_query = specific_project_url + '/files/trajectory.xtc'
print('We query the API at ' + trajectory_query)

# Download the file with an arbitrary name
trajectory_filename = f'{accession}_trajectory.xtc'

# Query the API
urllib.request.urlretrieve(trajectory_query, trajectory_filename)

if exists(trajectory_filename):
    print(f'File has been downloaded successfully ({trajectory_filename})')

***
4) **Trajectory visualization**
***

In [ ]:
# Show trajectory
try:
    import simpletraj
    view2 = nglview.show_simpletraj(
        nglview.SimpletrajTrajectory(trajectory_filename, structure_filename), gui=True)
    view2._remote_call('setSize', target='Widget', args=['800px','600px'])
    view2
except ImportError:
    import mdtraj as md, os as _os, py3Dmol
    from IPython.display import display as _display

    _t = md.load(trajectory_filename, top=structure_filename)
    _stride = max(1, _t.n_frames // 100)
    _t_sub = _t[::_stride]
    print(f'{_t.n_frames} frames → animating {_t_sub.n_frames} (stride {_stride})')

    _traj_pdb = _os.path.splitext(trajectory_filename)[0] + '_traj.pdb'
    _t_sub.save_pdb(_traj_pdb)
    with open(_traj_pdb) as _f:
        _pdb_content = _f.read()

    view2 = py3Dmol.view(width=800, height=600)
    view2.addModelsAsFrames(_pdb_content)
    view2.setStyle({'stick': {}})
    view2.zoomTo()
    view2.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
    view2.show()


***
5) **Trajectory analyses** (MDDB)
***

In [ ]:
# Retrieve the MDDB accession from the specific ABDB compound
mddb_entry_request = api_url + f'/compounds/{accession}'

# Query the API
with urllib.request.urlopen(mddb_entry_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

# Print link to MDDB MD analyses
compound_title = f"{parsed_response['Family'].title()}-{parsed_response['Compound'].title()} ({parsed_response['Charge']})"
mddb_link = f"MDDB (MDposit) link to MD analyses: [link={mddb_web}/#/id/{parsed_response['accession']}]{mddb_web}/#/id/{parsed_response['accession']}[/link]"
print(Panel(f"[bold blue]{compound_title}[/bold blue]\n\n{mddb_link}"))


<a id="qm"></a>
## QM data from AB-DB

**Downloading** and **exploring** **QM data** from the **AB-DB** entries

1) Download the **QM-optimized** structure and compare it against the original one
2) **AMBER force-field parameters** generated from the **QM calculations**
3) **Geometry optimization** and **Single-point energy calculation** data (ioChem-BD)

***
1) Download the **QM-optimized** structure and compare it against the original one
***

In [ ]:
qm_query = f"{api_url}/compounds/{accession}/files/qm_optimized.sdf"

# Download the file with an arbitrary name
qm_filename = f'{accession}_qm_optimized.sdf'

# Query the API
urllib.request.urlretrieve(qm_query, qm_filename)

if exists(qm_filename):
    print(f'File has been downloaded successfully ({qm_filename})')

In [ ]:
view = nglview.show_structure_file(qm_filename)
view._remote_call('setSize', target='Widget', args=['','600px'])
view

In [ ]:
# Show structures: protein, ligand and protein-ligand complex
view1 = nglview.show_structure_file(structure_filename)
view1._remote_call('setSize', target='Widget', args=['550px','400px'])
view1.camera='orthographic'
view1
view2 = nglview.show_structure_file(qm_filename)
view2.add_representation(repr_type='ball+stick')
view2._remote_call('setSize', target='Widget', args=['550px','400px'])
view2.camera='orthographic'
view2
ipywidgets.HBox([view1, view2])

***
2) **AMBER force-field parameters** generated from the **QM calculations**
***

In [ ]:
# AMBER frcmod
ff_frcmod_query = f"{api_url}/compounds/{accession}/files/ff_mol.frcmod"

# Download the file with an arbitrary name
ff_frcmod_filename = f'{accession}_ff.frcmod'

# Query the API
urllib.request.urlretrieve(ff_frcmod_query, ff_frcmod_filename)

if not exists(ff_frcmod_filename):
    print(f'File has not been downloaded successfully ({ff_frcmod_filename}). Please check!')

# AMBER prep file
ff_prep_query = f"{api_url}/compounds/{accession}/files/ff_mol.prep"

# Download the file with an arbitrary name
ff_prep_filename = f'{accession}_ff.prep'

# Query the API
urllib.request.urlretrieve(ff_prep_query, ff_prep_filename)

if not exists(ff_prep_filename):
    print(f'File has not been downloaded successfully ({ff_prep_filename}). Please check!')

frcmod_url = f"/files/{ff_frcmod_filename}"
prep_url = f"/files/{ff_prep_filename}"

print(
    Panel(
        f"[bold]AMBER frcmod file:[/bold]\n\n[link={frcmod_url}]{ff_frcmod_filename}[/link]",
        border_style="grey37"
    ),
    Panel(
        f"[bold]AMBER prep file:[/bold]\n\n[link={prep_url}]{ff_prep_filename}[/link]",
        border_style="grey37"
    )
)

***
3) **Geometry optimization** and **Single-point energy calculation** data (ioChem-BD)
***

In [ ]:
# Retrieve the MDDB accession from the specific ABDB compound
mddb_entry_request = api_url + f'/compounds/{accession}'

# Query the API
with urllib.request.urlopen(mddb_entry_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

family = parsed_response['Family']
compound = parsed_response['Compound']
charge = parsed_response['Charge']

ioChem_url = f"{iochem_bd_web}/simple-search?query=%22{family}%20{compound}%20{charge}%22"

print(
    Panel(
        f"[bold]ioChem-BD QM entries:[/bold]\n\n[link={ioChem_url}]{ioChem_url}[/link]",
        border_style="grey37"
    )
)

***
<a id="questions"></a>

## Questions & Comments

Questions, issues, suggestions and comments are really welcome!

* GitHub issues:
    * [https://github.com/bioexcel/biobb](https://github.com/bioexcel/biobb)